# 03 - Data Cleaning and Standardization
### Credit Portfolio Segmentation and Risk Pattern Analysis

---

## Cleaning & Preparing the Data
Raw loan data often comes with formatting quirks, non-standard naming, and missing values. Here, we:
1. **Rename columns** to standard industry terms (`annual_income`, `loan_amount`, `dti`, `fico_score`).
2. **Impute missing numbers using the median** (resists extreme values).
3. **Filter out extreme outliers** (e.g. negative income or impossible debt ratios).
4. **Save our clean working dataset** to `../data/processed/cleaned_loans_dataset.csv`.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

raw_path = Path("../data/raw/loans_granting_model_dataset.csv")
if not raw_path.exists():
    raw_path = Path("data/raw/loans_granting_model_dataset.csv")

df = pd.read_csv(raw_path, nrows=25000)
print(f"Raw shape: {df.shape}")


---
## 1. Column Renaming & Feature Selection
We map short, raw database columns to clear business descriptions.


In [ ]:
column_mapping = {
    'revenue': 'annual_income',
    'loan_amnt': 'loan_amount',
    'dti_n': 'dti',
    'fico_n': 'fico_score',
    'experience_c': 'emp_length_years',
    'home_ownership_n': 'home_ownership'
}

df_clean = df.rename(columns=column_mapping)

columns_to_keep = [
    'annual_income', 'dti', 'loan_amount', 'fico_score', 
    'emp_length_years', 'home_ownership', 'purpose', 'addr_state', 'Default'
]
df_clean = df_clean[[col for col in columns_to_keep if col in df_clean.columns]].copy()
print("Selected columns:", df_clean.columns.tolist())


---
## 2. Imputing Missing Values with the Median
Financial metrics such as income are right-skewed: a few ultra-high earners pull the average upward. The median reflects the true middle of the distribution and prevents distorted imputations.


In [ ]:
num_cols = ['annual_income', 'dti', 'loan_amount', 'fico_score', 'emp_length_years']

for col in num_cols:
    if col in df_clean.columns:
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)
        print(f"Imputed '{col}' with median: {median_val}")

print("\nNull counts after imputation:")
print(df_clean[num_cols].isnull().sum())


---
## 3. Filtering Unrealistic Outliers
We remove records that represent obvious data entry errors:
- Income under $5,000 or exceeding $1,000,000.
- Debt-to-Income (DTI) below 0% or above 100%.


In [ ]:
initial_len = len(df_clean)

df_clean = df_clean[(df_clean['annual_income'] >= 5000) & (df_clean['annual_income'] <= 1000000)]
df_clean = df_clean[(df_clean['dti'] >= 0) & (df_clean['dti'] <= 100)]

print(f"Filtered {initial_len - len(df_clean)} outlier rows. Clean dataset size: {len(df_clean):,}")


---
## 4. Saving the Cleaned Dataset


In [ ]:
output_path = Path("../data/processed/cleaned_loans_dataset.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(output_path, index=False)
print(f"Clean dataset saved to: {output_path.resolve()}")
df_clean.head(3)
